In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

from google.colab import files

uploaded = files.upload()

Saving reviews.csv to reviews.csv


In [2]:
# Загружаем датасет

df = pd.read_csv("reviews.csv")

# Первые строки датасета

df.head()

# Общая информация

df.info()

# Статистика

df.describe(include="all")

# Проверяем баланс классов

df["sentiment"].value_counts()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


,count
sentiment,
positive,25000
negative,25000


In [3]:
# Признаки (тексты отзывов)
X = df["review"]

# Целевая переменная (метки классов)
y = df["sentiment"]

# Разделение данных на обучающую и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Проверяем размеры выборок
print("Количество обучающих примеров:", len(X_train))
print("Количество тестовых примеров:", len(X_test))

Количество обучающих примеров: 40000
Количество тестовых примеров: 10000


In [4]:
# Создаем объект TF-IDF
vectorizer = TfidfVectorizer()

# Обучаем TF-IDF на обучающей выборке
X_train_tfidf = vectorizer.fit_transform(X_train)

# Преобразуем тестовую выборку
X_test_tfidf = vectorizer.transform(X_test)

# Проверяем размер полученных матриц
print("Размер обучающей матрицы:", X_train_tfidf.shape)
print("Размер тестовой матрицы:", X_test_tfidf.shape)

Размер обучающей матрицы: (40000, 93003)
Размер тестовой матрицы: (10000, 93003)


In [5]:
# Создаем модель логистической регрессии
model = LogisticRegression(max_iter=1000)

# Обучаем модель
model.fit(
    X_train_tfidf,
    y_train
)

print("Модель успешно обучена.")

Модель успешно обучена.


In [6]:
# Получаем предсказания модели для тестовой выборки
y_pred = model.predict(X_test_tfidf)

# Покажем первые 10 предсказаний
print(y_pred[:10])

['negative' 'positive' 'negative' 'positive' 'negative' 'positive'
 'positive' 'negative' 'negative' 'negative']


In [7]:
# Вычисляем основные метрики качества

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, pos_label="positive")
recall = recall_score(y_test, y_pred, pos_label="positive")
f1 = f1_score(y_test, y_pred, pos_label="positive")

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)

Accuracy: 0.8999
Precision: 0.8914307871267934
Recall: 0.9124826354435404
F1-score: 0.9018338727076591


In [8]:
# Полный отчет по классификации

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

    negative       0.91      0.89      0.90      4961
    positive       0.89      0.91      0.90      5039

    accuracy                           0.90     10000
   macro avg       0.90      0.90      0.90     10000
weighted avg       0.90      0.90      0.90     10000



In [9]:
# Вычисляем матрицу ошибок

cm = confusion_matrix(y_test, y_pred)

print(cm)

[[4401  560]
 [ 441 4598]]


In [10]:
import joblib

# Сохраняем обученную модель
joblib.dump(model, "sentiment_model.pkl")

# Сохраняем TF-IDF векторизатор
joblib.dump(vectorizer, "tfidf_vectorizer.pkl")

print("Модель и векторизатор успешно сохранены.")

Модель и векторизатор успешно сохранены.


In [11]:
# Проверяем модель на новых примерах

new_reviews = [
    "This movie is absolutely fantastic!",
    "I hated this film. It was boring and too long.",
    "The actors did a great job.",
    "Worst movie I have ever seen."
]

# Преобразуем текст в TF-IDF
new_reviews_tfidf = vectorizer.transform(new_reviews)

# Получаем предсказания
predictions = model.predict(new_reviews_tfidf)

# Выводим результат
for review, prediction in zip(new_reviews, predictions):
    print(f"Review: {review}")
    print(f"Prediction: {prediction}")
    print("-" * 60)

Review: This movie is absolutely fantastic!
Prediction: positive
------------------------------------------------------------
Review: I hated this film. It was boring and too long.
Prediction: negative
------------------------------------------------------------
Review: The actors did a great job.
Prediction: positive
------------------------------------------------------------
Review: Worst movie I have ever seen.
Prediction: negative
------------------------------------------------------------
